In [1]:
import dill
import scanpy as sc
import anndata as ad
import muon as mu
import os
import warnings
warnings.filterwarnings("ignore")
import pandas
import pyranges
# Set stderr to null to avoid strange messages from ray
import sys
_stderr = sys.stderr
null = open(os.devnull,'wb')

from pycisTopic.cistopic_class import CistopicObject
from pycisTopic.lda_models import *
from pycisTopic.clust_vis import *
from pycisTopic.diff_features import *

from scenicplus.scenicplus_class import SCENICPLUS, create_SCENICPLUS_object
from scenicplus.preprocessing.filtering import *
from scenicplus.utils import region_names_to_coordinates
from scenicplus.eregulon_enrichment import make_rankings

from scenicplus.wrappers.run_scenicplus import run_scenicplus

sys.path.append('/nfs/casm/team215mg/users/od5/scripts/')
from scenicplus_modified_dotplot import *

In [2]:
## Set up work directory
work_dir = '/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/scenicplus/tf_pilot/batch1/ipsc/'

## Tmp dir for ray (this needs to be as short as possible, as there is an overall limit of the number of characters accepted by ray - max is 107):
tmp_dir = '/lustre/scratch125/casm/teams/team215/r/'
os.makedirs(tmp_dir, exist_ok=True)

In [3]:
## Load objects 
mdata = mu.read_h5mu('/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/seacells_gex/tf_pilot/batch1/ipsc/seacells_annot_min10cells.h5mu')
adata = mdata['rna']

In [4]:
## Filter for expressed genes
mdata_single_cell = mu.read_h5mu('/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/seacells_gex/tf_pilot/batch1/ipsc/mdata.h5mu')
adata_single_cell = mdata_single_cell['rna']

# adata_single_cell.obs['anno_v1_nuc_gex'] = adata_single_cell.obs['anno_v1_nuc_gex'].str.replace('/', '_', regex=True)
# adata_single_cell.obs['anno_v1_nuc_gex'] = adata_single_cell.obs['anno_v1_nuc_gex'].astype('category')

## Estimate fraction of accessible cells in each group
def calculate_fraction(adata, slot, obs):
    more_than_zero = np.array((adata[adata.obs[slot] == obs, :].X > 0).mean(axis=0)).flatten()
    return pd.Series(more_than_zero , index=adata.var_names)

# Applying the function to each cell type x stim and combining results
groups = adata_single_cell.obs['perturbation'].unique()
results = {g: calculate_fraction(adata_single_cell, 'perturbation',g) for g in groups}

# Convert the results dictionary to a DataFrame
results_df = pd.DataFrame(results)

results_df.to_csv((work_dir + 'genes_f_exp_by_perturbation_min50cells.csv'))

In [5]:
adata_single_cell.obs['perturbation']

multi_Imm_NK15385635-AAACAGCCAGCACGAA-1     TCF7
multi_Imm_NK15385635-AAACAGCCATAATGAG-1    CEBPD
multi_Imm_NK15385635-AAACAGCCATTCAGCA-1     TCF7
multi_Imm_NK15385635-AAACATGCAAGGAATC-1    RUNX2
multi_Imm_NK15385635-AAACATGCATCGTTCT-1    CEBPD
                                           ...  
multi_Imm_NK15385635-TTTGTGTTCTGCAACG-1    RUNX2
multi_Imm_NK15385635-TTTGTGTTCTTGTCTG-1    TBX21
multi_Imm_NK15385635-TTTGTTGGTGCCTCAC-1     TCF7
multi_Imm_NK15385635-TTTGTTGGTGTCCAAA-1    FOXO3
multi_Imm_NK15385635-TTTGTTGGTTTATTCG-1    CEBPD
Name: perturbation, Length: 5400, dtype: category
Categories (9, object): ['Control' < 'CEBPD' < 'EBF1' < 'FOXO3' ... 'PRDM1' < 'RUNX2' < 'TBX21' < 'TCF7']

In [6]:
sum(results_df.max(axis=1) > 0.025)

16752

In [7]:
## Subsetting for genes expressed in at least 2.5% of cells in at least one group (cell type x stim)
highly_expressed_genes = results_df.index[results_df.max(axis=1) > 0.025]
adata = adata[:, adata.var_names.isin(highly_expressed_genes)].copy()

In [8]:
adata.X.sum(axis=1)

matrix([[9929.09150296],
        [9925.57720147],
        [9927.50783027],
        [9931.61694224],
        [9943.0925866 ],
        [9936.39928463],
        [9920.79062149],
        [9932.10326279],
        [9931.0706705 ],
        [9923.32319695],
        [9911.5567172 ],
        [9932.034325  ],
        [9936.47797683],
        [9931.24917618],
        [9935.96863256],
        [9924.00958895],
        [9933.17073802],
        [9923.98804769],
        [9924.21137753],
        [9938.43305686],
        [9931.05604161],
        [9911.47255548],
        [9923.77607227],
        [9936.32179445],
        [9933.57542807],
        [9916.23627255],
        [9941.54450374],
        [9925.13347496],
        [9926.64886989],
        [9937.45753072],
        [9933.79171645],
        [9917.51434418],
        [9929.03113636],
        [9919.28922531],
        [9897.93468874],
        [9940.95185614],
        [9915.28575425],
        [9939.8732145 ],
        [9941.10418371],
        [9931.23561897],


In [9]:
## Scenic+ requires integers, so rounding to the nearest integer
## To minimise the impact of rounding, we will first multiply by 10 (to 100,000 total)
adata.X = adata.X * 10
adata.X = np.rint(adata.X).astype(np.int32)

In [10]:
## Load cistopic and motif enrichment
cistopic_obj = dill.load(open((work_dir + '/cistopic/scATAC/cistopic_obj.pkl'), 'rb'))
menr = dill.load(open((work_dir + '/motifs/menr.pkl'), 'rb'))

In [11]:
menr

{'CTX_topics_otsu_All': {'Topic1': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef64d550>,
  'Topic2': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef654520>,
  'Topic3': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef658af0>,
  'Topic4': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef79f640>,
  'Topic5': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef5a3160>,
  'Topic6': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef590850>,
  'Topic7': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef451850>,
  'Topic8': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef4a4580>,
  'Topic9': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef4e8b50>,
  'Topic10': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef4d7100>,
  'Topic11': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150cef515df0>,
  'Topic12': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ce

In [12]:
menr['CTX_DARs_archr_All']

{'CEBPD': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab7adcac0>,
 'RUNX2': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab7a924c0>,
 'TBX21': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab7ac8d30>}

In [13]:
menr['CTX_DARs_archr_No_promoters']

{'CEBPD': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab708da60>,
 'RUNX2': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab6f792b0>,
 'TBX21': <pycistarget.motif_enrichment_cistarget.cisTarget at 0x150ab6f30670>}

In [14]:
## Create imputed scenicplus object
scplus_obj = create_SCENICPLUS_object(
    GEX_anndata = adata,
    cisTopic_obj = cistopic_obj,
    menr = menr
)
scplus_obj.X_EXP = np.array(scplus_obj.X_EXP.todense())
scplus_obj

2026-05-11 13:00:23,051 cisTopic     INFO     Imputing region accessibility
2026-05-11 13:00:23,055 cisTopic     INFO     Impute region accessibility for regions 0-20000
2026-05-11 13:00:23,156 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2026-05-11 13:00:23,215 cisTopic     INFO     Done!


SCENIC+ object with n_cells x n_genes = 246 x 16752 and n_cells x n_regions = 246 x 36627
	metadata_genes:'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
	metadata_cell:'GEX_perturbation', 'GEX_perturbation_purity', 'GEX_n_genes_by_counts', 'GEX_log1p_n_genes_by_counts', 'GEX_total_counts', 'GEX_log1p_total_counts', 'GEX_pct_counts_in_top_50_genes', 'GEX_pct_counts_in_top_100_genes', 'GEX_pct_counts_in_top_200_genes', 'GEX_pct_counts_in_top_500_genes', 'ACC_rna:perturbation', 'ACC_rna:perturbation_purity', 'ACC_rna:n_genes_by_counts', 'ACC_rna:log1p_n_genes_by_counts', 'ACC_rna:total_counts', 'ACC_rna:log1p_total_counts', 'ACC_rna:pct_counts_in_top_50_genes', 'ACC_rna:pct_counts_in_top_100_genes', 'ACC_rna:pct_counts_in_top_200_genes', 'ACC_rna:pct_counts_in_top_500_genes', 'ACC_atac:perturbation', 'ACC_atac:perturbation_purity', 'ACC_perturbation', 'ACC_perturbation_purity', 'ACC_compactness', 'ACC_size'
	menr:'CTX

In [15]:
## Cleanup some files
del mdata
del adata
del cistopic_obj
del mdata_single_cell
del adata_single_cell
del menr

In [16]:
run_name = 'scenicplus_full'
grouping_variable = 'GEX_perturbation'

os.makedirs((work_dir + run_name), exist_ok=True)

In [17]:
## Export
dill.dump(scplus_obj, open((work_dir + run_name + '/scplus_obj.pkl'), 'wb'))

In [18]:
## Submit job to the cluster (from within the multiome conda environment)
print('cd ' + os.getcwd())
print('conda activate multiome')
for i in range(1,21):
    print('/nfs/casm/team215mg/users/od5/scripts/jsub lsf -n ' + run_name + '_' + str(i) + ' -q long -c 16 -m 200g "python scenicplus_subsample_wrapper.py --work_dir ' + \
          work_dir + run_name + '/ --grouping_variable ' + grouping_variable + ' --ncpu 16 --seed ' + str(i) + ' --fraction 0.8" | bsub -G team215-grp')

cd /nfs/casm/team215mg/users/od5/projects/celltherapy/ipsc/multiome/tf_pilot/005_GRNs/001_ipsc
conda activate multiome
/nfs/casm/team215mg/users/od5/scripts/jsub lsf -n scenicplus_full_1 -q long -c 16 -m 200g "python scenicplus_subsample_wrapper.py --work_dir /lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/scenicplus/tf_pilot/batch1/ipsc/scenicplus_full/ --grouping_variable GEX_perturbation --ncpu 16 --seed 1 --fraction 0.8" | bsub -G team215-grp
/nfs/casm/team215mg/users/od5/scripts/jsub lsf -n scenicplus_full_2 -q long -c 16 -m 200g "python scenicplus_subsample_wrapper.py --work_dir /lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/scenicplus/tf_pilot/batch1/ipsc/scenicplus_full/ --grouping_variable GEX_perturbation --ncpu 16 --seed 2 --fraction 0.8" | bsub -G team215-grp
/nfs/casm/team215mg/users/od5/scripts/jsub lsf -n scenicplus_full_3 -q long -c 16 -m 200g "python scenicplus_subsample_wrapper.py --work_dir /lustr

In [42]:
## We can explore the results in the next notebook

In [19]:
scplus_obj.metadata_cell

,GEX_perturbation,GEX_perturbation_purity,GEX_n_genes_by_counts,GEX_log1p_n_genes_by_counts,GEX_total_counts,GEX_log1p_total_counts,GEX_pct_counts_in_top_50_genes,GEX_pct_counts_in_top_100_genes,GEX_pct_counts_in_top_200_genes,GEX_pct_counts_in_top_500_genes,...,ACC_rna:pct_counts_in_top_50_genes,ACC_rna:pct_counts_in_top_100_genes,ACC_rna:pct_counts_in_top_200_genes,ACC_rna:pct_counts_in_top_500_genes,ACC_atac:perturbation,ACC_atac:perturbation_purity,ACC_perturbation,ACC_perturbation_purity,ACC_compactness,ACC_size
FOXO3-SEACell-7,FOXO3,1.0,13377,9.501367,56233.599900,10.937288,3.247382,5.563213,9.582435,19.336516,...,3.247382,5.563213,9.582435,19.336516,FOXO3,1.0,FOXO3,1.0,4.617233e-01,13
EBF1-SEACell-1,EBF1,1.0,12668,9.446913,53891.578698,10.894748,3.392604,5.805529,9.851596,19.627359,...,3.392604,5.805529,9.851596,19.627359,EBF1,1.0,EBF1,1.0,3.685451e-01,13
FOXO3-SEACell-0,FOXO3,1.0,16927,9.736724,212205.625358,12.265316,3.380248,5.845951,10.115622,19.963238,...,3.380248,5.845951,10.115622,19.963238,FOXO3,1.0,FOXO3,1.0,1.609073e+01,51
Control-SEACell-30,Control,1.0,12204,9.409601,44581.291144,10.705092,3.518525,5.991696,10.136075,20.073998,...,3.518525,5.991696,10.136075,20.073998,Control,1.0,Control,1.0,2.590334e-08,11
Control-SEACell-7,Control,1.0,14936,9.611597,74575.866413,11.219586,3.147421,5.437560,9.377975,18.983404,...,3.147421,5.437560,9.377975,18.983404,Control,1.0,Control,1.0,1.433780e+00,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HOPX-SEACell-21,HOPX,1.0,14942,9.611998,109002.255781,11.599133,3.414740,5.942261,10.273339,20.117645,...,3.414740,5.942261,10.273339,20.117645,HOPX,1.0,HOPX,1.0,9.055228e-01,26
HOPX-SEACell-9,HOPX,1.0,15892,9.673634,110358.569100,11.611499,3.191632,5.489531,9.511860,19.150202,...,3.191632,5.489531,9.511860,19.150202,HOPX,1.0,HOPX,1.0,3.223938e-01,25
Control-SEACell-18,Control,1.0,16216,9.693815,149775.450140,11.916899,3.206128,5.552622,9.598010,19.396749,...,3.206128,5.552622,9.598010,19.396749,Control,1.0,Control,1.0,7.802440e-01,35
TCF7-SEACell-40,TCF7,1.0,12202,9.409437,41838.215301,10.641589,3.184363,5.419519,9.310157,18.795018,...,3.184363,5.419519,9.310157,18.795018,TCF7,1.0,TCF7,1.0,5.084981e+00,10
